<a href="https://colab.research.google.com/github/engelberger/frustrapy/blob/dev_plots/frustrapy_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Frustratometer in Python
<a href="https://colab.research.google.com/github/engelberger/frustrapy/blob/main/FrustraPy_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Introduction

The concept of frustration in proteins refers to the presence of conflicting interactions within the protein structure. These conflicts arise when the local interactions within a protein are not optimally stabilizing, leading to a certain degree of energetic frustration. The following is a summary please refer to the original paper for more details:

[Protein Frustratometer 2: a tool to localize energetic frustration in protein molecules, now with electrostatics](https://academic.oup.com/nar/article/44/W1/W356/2499321)
[FrustrometerR: an R-package to compute local frustration in protein structures, point mutants and MD simulations](https://academic.oup.com/bioinformatics/article/37/18/3038/6171179)


## There are three main types of frustration

* *Highly frustrated*: Highly frustrated regions in a protein are those where the local interactions are significantly destabilizing compared to what would be expected in an idealized, energetically minimized structure. These regions often play crucial roles in protein function, such as binding sites, allosteric sites, or regions involved in conformational changes. For example, in an enzyme, the active site might be highly frustrated to allow for substrate binding and catalysis, which require a certain degree of flexibility and adaptability.
* *Neutral*: Neutral regions in a protein are those where the local interactions are neither significantly stabilizing nor destabilizing. These regions may not directly contribute to protein function but are essential for maintaining the overall structural integrity of the protein. Neutral regions can serve as a buffer between highly frustrated and minimally frustrated regions, allowing for the necessary flexibility and stability balance within the protein.
* *Minimally frustrated*: Minimally frustrated regions in a protein are those where the local interactions are highly optimized and stabilizing. These regions typically form the stable core of the protein and are essential for maintaining the native folded state. Minimally frustrated regions often consist of hydrophobic residues that pack tightly together, forming a stable foundation for the protein structure. For example, in the case of globular proteins, the hydrophobic core is usually minimally frustrated, contributing to the overall stability of the folded state.

## Significance of Frustration in Proteins:

* Protein folding: During the protein folding process, the polypeptide chain navigates through an energy landscape to reach its native state. The concept of minimal frustration suggests that evolution has optimized the folding landscape to minimize energetic conflicts, allowing proteins to fold efficiently and avoid getting trapped in non-native states.
* Allostery: Allosteric regulation in proteins often involves highly frustrated regions that undergo conformational changes upon ligand binding or other perturbations. These frustrated regions allow for the propagation of allosteric signals throughout the protein structure, enabling long-range communication and regulation of protein function.
* Protein-protein interactions: Protein interfaces often contain a mix of highly frustrated and minimally frustrated regions. The highly frustrated regions may contribute to the specificity and adaptability of the interaction, while the minimally frustrated regions provide stability to the complex. The balance between frustration and stability at the interface is crucial for the formation and regulation of protein complexes.

In [ ]:
# @title Install
%cd /content
%pip install -q biopython igraph leidenalg
%pip -q install git+https://github.com/engelberger/frustrapy.git@dev_plots
%pip install -q -U kaleido==0.2.1
%pip -q install py3dmol


In [ ]:
# @title Frustratometer in Python {"display-mode":"form"}

# @markdown ### General Settings
# @markdown Select the analysis mode:
# @markdown - `configurational`: Calculates frustration index for the whole structure.
# @markdown - `singleresidue`: Calculates the frustration when mutating residues to Alanine (or Glycine if Alanine). Can analyze all residues or a specific subset (see options below).
# @markdown - `mutational`: Calculates the frustration when mutating specific residues provided by the user (requires extra setup).
mode = "configurational" # @param ["configurational", "singleresidue", "mutational"]

# @markdown ---
# @markdown ### Input Data Options

# @markdown **Option 1: Use Example PDBs**
# @markdown Check this box to download and use example PDB files (1fhj, 2dn1, 1m6k).
# @markdown This will ignore the `pdbs_dir` and `pdb_id` settings below.
example = True # @param {type:"boolean"}

# @markdown **Option 2: Specify a PDB ID** (Only used if 'example' is unchecked)
# @markdown Enter a 4-character PDB ID (e.g., 1l2y) to download from RCSB PDB.
# @markdown The file will be saved in the `pdbs_dir`.
pdb_id = "" # @param {type:"string"}

# @markdown **Option 3: Specify a Directory** (Only used if 'example' is unchecked and 'pdb_id' is empty)
# @markdown Provide the path to a directory containing your PDB files.
pdbs_dir = "/content" # @param {type:"string"}

# @markdown ---
# @markdown ### Single Residue Mode Options (Only used if mode = 'singleresidue')
# @markdown Specify the target chain and optionally a list of residue numbers to analyze.
# @markdown If 'Target Residues' is left empty, all residues in the target chain will be analyzed.

# @markdown **Target Chain:**
# @markdown Enter the chain ID (e.g., A) for single residue analysis.
target_chain = "A" # @param {type:"string"}

# @markdown **Target Residues (comma-separated):**
# @markdown Enter a comma-separated list of residue numbers (e.g., 18,19,50) to analyze within the target chain.
# @markdown Leave empty to analyze all residues in the chain.
target_residues_csv = "" # @param {type:"string"}

# @markdown ---
# @markdown ### Output and Execution Settings

# @markdown **Results Directory:**
# @markdown Specify the directory where the analysis results and plots will be saved.
# @markdown If 'example' is checked, results go to '/content/Results_example'.
results_dir = "/content/Results" # @param {type:"string"}

# @markdown **Overwrite Results:**
# @markdown If checked, any existing results in the target results directory will be deleted before the analysis runs.
overwrite = False # @param {type:"boolean"}

# @markdown **Debug Mode:**
# @markdown Enable debug mode for more verbose output (useful for troubleshooting).
debug = False # @param {type:"boolean"}

# @markdown ---

import sys
import os
import frustrapy
import re # Import regex for input validation

# --- Input Validation and Setup --- 

# Validate mode selection
allowed_modes = ["configurational", "singleresidue", "mutational"]
if mode not in allowed_modes:
    print(f"Error: Invalid mode '{mode}'. Please choose from {allowed_modes}.", file=sys.stderr)
    sys.exit(1)

# Validate PDB ID format if provided
if not example and pdb_id:
    pdb_id = pdb_id.strip().lower()
    if not re.match(r"^[a-zA-Z0-9]{4}$", pdb_id):
        print(f"Error: Invalid PDB ID format '{pdb_id}'. Please provide a 4-character ID.", file=sys.stderr)
        sys.exit(1)

# Validate target_chain if mode is singleresidue
if mode == "singleresidue":
    target_chain = target_chain.strip()
    if not target_chain:
        print(f"Error: Target Chain cannot be empty for singleresidue mode.", file=sys.stderr)
        sys.exit(1)
    # Basic check if it looks like a chain ID (can be multi-character)
    # if not re.match(r"^[a-zA-Z0-9]+$", target_chain):
    #     print(f"Error: Invalid Target Chain format '{target_chain}'.", file=sys.stderr)
    #     sys.exit(1)

# Parse target_residues_csv if provided for singleresidue mode
residues_to_analyze_dict = None
if mode == "singleresidue" and target_residues_csv:
    target_residues_csv = target_residues_csv.strip()
    try:
        residue_list = [int(res.strip()) for res in target_residues_csv.split(',') if res.strip()]
        if not residue_list:
             print(f"Warning: Target Residues list is empty after processing '{target_residues_csv}'. Analyzing all residues in chain {target_chain}.")
        else:
            residues_to_analyze_dict = {target_chain: residue_list}
            print(f"Parsed target residues for chain {target_chain}: {residue_list}")
    except ValueError as e:
        print(f"Error: Invalid format in Target Residues '{target_residues_csv}'. Please provide comma-separated integers. Details: {e}", file=sys.stderr)
        sys.exit(1)

# --- Determine Input/Output Directories --- 

current_results_dir = results_dir # Default
pdbs_to_process_dir = pdbs_dir # Default

# If the example is True, override paths and download examples
if example:
    print("Example is True. Downloading example PDB files...")
    pdbs_to_process_dir = "/content" # Use fixed dir for examples
    current_results_dir = "/content/Results_example" # Use fixed results dir for examples

    try:
        # Use shell commands directly without markdown interference
        get_ipython().system(f'wget -q https://files.rcsb.org/download/1fhj.pdb -O {os.path.join(pdbs_to_process_dir, "1fhj.pdb")}')
        get_ipython().system(f'wget -q https://files.rcsb.org/download/2dn1.pdb -O {os.path.join(pdbs_to_process_dir, "2dn1.pdb")}')
        get_ipython().system(f'wget -q https://files.rcsb.org/download/1m6k.pdb -O {os.path.join(pdbs_to_process_dir, "1m6k.pdb")}')
        print(f"Successfully downloaded example PDBs to {pdbs_to_process_dir}")
    except Exception as e:
        print(f"Error downloading example PDBs: {e}", file=sys.stderr)
        sys.exit(1)

    print(f"Using example PDBs from: {pdbs_to_process_dir}")
    print(f"Results will be saved to: {current_results_dir}")

# If example is False and a pdb_id is provided, download that PDB
elif pdb_id: # pdb_id is already validated and lowercased
    print(f"Example is False, PDB ID provided. Downloading PDB: {pdb_id}")
    pdbs_to_process_dir = pdbs_dir # PDB will be downloaded here
    current_results_dir = results_dir # Use user-specified results dir
    pdb_filename = f"{pdb_id}.pdb"
    output_path = os.path.join(pdbs_to_process_dir, pdb_filename)
    download_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

    # Ensure download directory exists
    os.makedirs(pdbs_to_process_dir, exist_ok=True)

    # Download the specified PDB file using get_ipython().system
    print(f"Downloading from {download_url} to {output_path}...")
    try:
        get_ipython().system(f'wget -q {download_url} -O "{output_path}"')
        # Check if download was successful (basic check: file exists and has size > 0)
        if not os.path.exists(output_path) or os.path.getsize(output_path) == 0:
            print(f"Error: Failed to download or received empty PDB file for ID: {pdb_id}. Please check the ID and internet connection.", file=sys.stderr)
            # Clean up empty file if it exists
            if os.path.exists(output_path): os.remove(output_path)
            sys.exit(1)
        else:
            print(f"Successfully downloaded {pdb_filename} to {pdbs_to_process_dir}")
    except Exception as e:
        print(f"Error during PDB download: {e}", file=sys.stderr)
        sys.exit(1)

    print(f"Using downloaded PDB from: {pdbs_to_process_dir}")
    print(f"Results will be saved to: {current_results_dir}")

# If example is False and no pdb_id is provided, use the specified pdbs_dir
else:
    print(f"Example is False, no PDB ID provided.")
    pdbs_to_process_dir = pdbs_dir
    current_results_dir = results_dir
    # Validate the specified pdbs_dir
    if not os.path.isdir(pdbs_to_process_dir):
         print(f"Error: Specified Input PDB directory '{pdbs_to_process_dir}' does not exist or is not a directory.", file=sys.stderr)
         sys.exit(1)
    # Check if directory contains PDB files
    try:
        pdb_files_found = [f for f in os.listdir(pdbs_to_process_dir) if f.lower().endswith(('.pdb', '.pdb.gz', '.cif', '.cif.gz'))]
        if not pdb_files_found:
            print(f"Warning: No PDB or mmCIF files found in the specified input directory '{pdbs_to_process_dir}'.", file=sys.stderr)
            # Decide if you want to exit or proceed
            # sys.exit(1) # Optionally exit if the directory is empty or has no structure files
        else:
            print(f"Found {len(pdb_files_found)} potential structure file(s) in {pdbs_to_process_dir}.")
    except OSError as e:
        print(f"Error accessing input PDB directory '{pdbs_to_process_dir}': {e}", file=sys.stderr)
        sys.exit(1)

    print(f"Using existing structure files from: {pdbs_to_process_dir}")
    print(f"Results will be saved to: {current_results_dir}")


# --- Handle Overwrite --- 

abs_results_dir = os.path.abspath(current_results_dir)
if overwrite:
    print(f"Overwrite is True. Removing previous results from {abs_results_dir}...")
    if os.path.isdir(abs_results_dir):
        try:
            # Use get_ipython().system for robust shell commands in Colab/IPython
            get_ipython().system(f'rm -rf "{abs_results_dir}"/*')
            print(f"Successfully cleared {abs_results_dir}")
        except Exception as e:
            print(f"Error removing files from {abs_results_dir}: {e}", file=sys.stderr)
            # Decide if this is critical
            # sys.exit(1)
    else:
      print(f"Results directory {abs_results_dir} did not exist, nothing to remove.")

# Ensure the final results directory exists before running the analysis
try:
    os.makedirs(abs_results_dir, exist_ok=True)
except OSError as e:
    print(f"Error creating results directory '{abs_results_dir}': {e}", file=sys.stderr)
    sys.exit(1)


# --- Prepare Frustration Analysis Arguments --- 

print("\n" + "="*30)
print(" Preparing Frustratometer Analysis ")
print("="*30)
print(f"Mode: {mode}")
print(f"Input Structure(s) location: {os.path.abspath(pdbs_to_process_dir)}")
print(f"Results location: {abs_results_dir}")
print(f"Overwrite: {overwrite}")
print(f"Debug: {debug}")

# Base arguments for the frustratometer
frustration_args = {
    "pdbs_dir": pdbs_to_process_dir,
    "mode": mode,
    "results_dir": abs_results_dir,
    "debug": debug
}

# Add specific arguments for singleresidue mode if residues were provided
if mode == "singleresidue":
    print(f"Target Chain: {target_chain}")
    if residues_to_analyze_dict:
        frustration_args["chain"] = target_chain
        frustration_args["residues"] = residues_to_analyze_dict
        print(f"Target Residues: {residues_to_analyze_dict[target_chain]}")
    else:
        # If no specific residues, frustrappy should analyze all in the chain by default
        # However, dir_frustration might still require the 'chain' argument depending on its implementation
        frustration_args["chain"] = target_chain # Pass the chain anyway
        print(f"Target Residues: Analyzing all residues in chain {target_chain}")

print("="*30 + "\n")

# --- Run Analysis --- 

print("\n" + "="*30)
print(" Starting Frustratometer Analysis ")
print("="*30 + "\n")

try:
    # Run the analysis using keyword arguments
    plots_dir_dict = frustrapy.dir_frustration(**frustration_args)

    print("\n" + "="*30)
    print(" Frustratometer Analysis Complete ")
    print("="*30)
    if plots_dir_dict:
        print("Analysis finished. Results and plots (if generated) are in:")
        # Print unique parent directories
        unique_parent_dirs = set(os.path.dirname(plot_dir) for plot_dir in plots_dir_dict.values())
        for parent_dir in sorted(list(unique_parent_dirs)):
             print(f" - {parent_dir}")
        # Optional: List individual plot dirs if needed
        # for pdb, plot_dir in plots_dir_dict.items():
        #     print(f" - {pdb}: {plot_dir}")
    else:
        print("Analysis finished, but no plot directories were reported.")
        print(f"Check the results directory for output files: {abs_results_dir}")
        if not debug:
            print("Consider running with debug=True for more detailed logs.")
    print("="*30 + "\n")

except FileNotFoundError as e:
    print("\n" + "="*30, file=sys.stderr)
    print(" An Error Occurred: File Not Found ", file=sys.stderr)
    print("="*30, file=sys.stderr)
    print(f"Error Details: {e}", file=sys.stderr)
    print("Please ensure the input PDB directory and files exist and are accessible.", file=sys.stderr)
    print(f"Checked directory: {os.path.abspath(pdbs_to_process_dir)}", file=sys.stderr)
    print("="*30 + "\n", file=sys.stderr)

except KeyError as e:
    print("\n" + "="*30, file=sys.stderr)
    print(" An Error Occurred: Missing Data (KeyError) ", file=sys.stderr)
    print("="*30, file=sys.stderr)
    print(f"Error Details: Missing key {e}", file=sys.stderr)
    print("This might indicate an issue with the PDB file structure (e.g., missing chain or residues) or the analysis results.", file=sys.stderr)
    if mode == "singleresidue":
        print(f"Ensure chain '{target_chain}' exists in the PDB file(s).", file=sys.stderr)
        if residues_to_analyze_dict:
             print(f"Ensure residues {residues_to_analyze_dict[target_chain]} exist in chain '{target_chain}'.", file=sys.stderr)
    print("="*30 + "\n", file=sys.stderr)

except Exception as e:
    # Catch any other unexpected errors
    print("\n" + "="*30, file=sys.stderr)
    print(" An Unexpected Error Occurred During Analysis ", file=sys.stderr)
    print("="*30, file=sys.stderr)
    print(f"Error Type: {type(e).__name__}", file=sys.stderr)
    print(f"Error Details: {e}", file=sys.stderr)
    print("Please check the input files, parameters, and consider running with debug=True.", file=sys.stderr)
    print("Check the console output/logs above for more specific error messages from Frustratometer.", file=sys.stderr)
    print("="*30 + "\n", file=sys.stderr)
    # Optionally re-raise the exception if needed for Colab's error reporting
    # raise e 

### Plots

In [ ]:
import plotly.graph_objects as go

# Access the first element of the tuple, which is the dictionary of plots
plot_data_dict = plots_dir_dict[0]

# Iterate through the PDB IDs and their corresponding plots
for pdb_id, plot_types in plot_data_dict.items():
    for plot_type, fig in plot_types.items():
        # Wrap the figure using go.FigureWidget
        fig.show()

In [ ]:
import frustrapy
import os
import logging
logger = logging.getLogger(__name__)

logger.setLevel(logging.INFO)
from frustrapy.analysis.frustration_calculator import FrustrationCalculator

mode = "configurational"

# set the logger level to debug
pdb_file = "/content/2dn1.pdb"
pdb_id = None
chain = None
residues = None
electrostatics_k = 12
seq_dist = 12
graphics = True
visualization = True
results_dir = "/content/jupyter_results"
debug = True

In [ ]:

# Set flag for mutation calculations to suppress logging
is_mutation_calculation = mode == "singleresidue" and residues is not None
# Only log protocol for main calculations, not individual mutations
if is_mutation_calculation:
    logger.debug(f"\nRunning Frustration Protocol:")
    logger.debug(f"- Analysis Mode: {mode}")
    if pdb_file:
        logger.debug(f"- Input Structure: {os.path.basename(pdb_file)}")
    if chain:
        logger.debug(f"- Analyzing Chain(s): {chain}")
    if residues:
        for chain_id, res_list in residues.items():
            logger.debug(f"- Residues for Chain {chain_id}: {res_list}")
    logger.info(f"- Sequence Distance: {seq_dist}")
    if electrostatics_k is not None:
        logger.debug(f"- Electrostatics K: {electrostatics_k}")
    logger.debug(f"- Graphics Generation: {'Enabled' if graphics else 'Disabled'}")
    logger.debug(
        f"- Structure Visualization: {'Enabled' if visualization else 'Disabled'}\n"
    )


    logger.debug("Starting frustration calculation")
# Validate PDB file existence if provided
if pdb_file is not None:
    pdb_file = os.path.abspath(pdb_file)
    logger.debug(f"Using PDB file: {pdb_file}")
    if not os.path.exists(pdb_file):
        logger.error(f"PDB file not found: {pdb_file}")
        raise FileNotFoundError(f"PDB file not found: {pdb_file}")
# Make results_dir absolute path if provided
if results_dir is not None:
    results_dir = os.path.abspath(results_dir)
    logger.debug(f"Using results directory: {results_dir}")
logger.debug(f"Initializing FrustrationCalculator with mode: {mode}")



In [ ]:
calculator = FrustrationCalculator(
    pdb_file=pdb_file,
    pdb_id=pdb_id,
    chain=chain,
    residues=residues,
    electrostatics_k=electrostatics_k,
    seq_dist=seq_dist,
    mode=mode,
    graphics=graphics,
    visualization=visualization,
    results_dir=results_dir,
    debug=debug,
    is_mutation_calculation=is_mutation_calculation,
)
logger.debug("Starting calculation")
pdb_configurational, plots, density_results = calculator.calculate()
logger.debug("Calculation completed")

In [ ]:
from frustrapy.analysis.mutations import mutate_res_parallel, mutate_res
# set log level to debug
logger.setLevel(logging.INFO)
mutate_res_parallel(pdb=pdb_configurational, res_num=109, chain="A", split=True, method="threading")

In [ ]:
from frustrapy.visualization import plot_mutate_res
fig = plot_mutate_res(pdb=pdb_configurational, res_num=109, chain="A", save=True)
# set plotly render to vscode from plotly io
import plotly.io as pio
pio.renderers.default = "colab"
fig.show()

In [ ]:
%load_ext autoreload
%autoreload 2


# Import the visualization function
from frustrapy.visualization.structure import view_config_contacts_py3dmol

# Visualize contacts for ALA_109-F using your existing pdb object
view_config_contacts_py3dmol(
    pdb=pdb_configurational,  # Your existing Pdb object
    central_chain="A",        # The chain of interest
    central_res=109,          # The residue number
    width=800,                # Optional: viewer width (default: 800)
    height=600                # Optional: viewer height (default: 600)
)

## Single Residue Frustration

In [ ]:
# @title Frustratometer in Python {"display-mode":"form"}
mode = "singleresidue" # @param ["configurational", "singleresidue", "mutational"]
pdbs_dir = "/content" # @param {type:"string"}
results_dir = "/content/Results" # @param {type:"string"}
example = True # @param {type:"boolean"}
overwrite = False # @param {type:"boolean"}
debug = False # @param {type:"boolean"}

import sys
import os
import frustrapy
import pickle

# Define residues to analyze
residues_to_analyze = {"A": [18, 19]}

# If the example is True, we will download the example files
if example:
    !wget -q http://www.rcsb.org/pdb/files/1fhj.pdb -O 1fhj.pdb
    !wget -q http://www.rcsb.org/pdb/files/2dn1.pdb -O 2dn1.pdb
    !wget -q http://www.rcsb.org/pdb/files/1m6k.pdb -O 1m6k.pdb

    pdbs_dir = "/content"
    results_dir = "/content/Results_example"
    # Remove any previous results
    !rm -rf /content/Results_example/*

if overwrite:
    if example:
        !rm -rf /content/Results/*
    else:
        # Convert the results_dir to an absolute path
        results_dir = os.path.abspath(results_dir)
        os.system(f"rm -rf {results_dir}/*")

# Directory frustration analysis with specific residues
plots_dir_dict = frustrapy.dir_frustration(
    pdbs_dir=pdbs_dir,
    mode=mode,
    results_dir=results_dir,
    debug=debug,
    chain="A",
    residues=residues_to_analyze
)

# Analyze and display results
results_found = 0

for root, dirs, files in os.walk(results_dir):
    for file in files:
        if file.endswith("_single_residue_data.pkl"):
            pkl_path = os.path.join(root, file)
            with open(pkl_path, "rb") as f:
                data = pickle.load(f)
            print(f"\nAnalysis results from: {os.path.basename(pkl_path)}")
            results_found += 1

            if "A" in data:
                for res_num in [18, 19]:
                    if res_num in data["A"]:
                        res_data = data["A"][res_num]
                        mutations = res_data.mutations

                        # Find most and least frustrated mutations
                        most_frustrated = min(mutations.items(), key=lambda x: x[1])
                        least_frustrated = max(mutations.items(), key=lambda x: x[1])

                        print(f"\nPosition {res_num} (Native: {res_data.residue_name})")
                        print(f"Most frustrated mutation: {res_data.residue_name} → {most_frustrated[0]} "
                              f"(Frustration Index: {most_frustrated[1]:.3f})")
                        print(f"Least frustrated mutation: {res_data.residue_name} → {least_frustrated[0]} "
                              f"(Frustration Index: {least_frustrated[1]:.3f})")

                        # Sort and display mutations
                        sorted_mutations = sorted(mutations.items(), key=lambda x: x[1])
                        print("\nAll mutations sorted by frustration (top 5 most and least frustrated):")
                        print("Most frustrated:")
                        for mut, score in sorted_mutations[:5]:
                            print(f"  {res_data.residue_name} → {mut}: {score:.3f}")
                        print("Least frustrated:")
                        for mut, score in sorted_mutations[-5:]:
                            print(f"  {res_data.residue_name} → {mut}: {score:.3f}")
                        print("-" * 50)

In [ ]:
# Show the fig objects
for pdb in plots_dir_dict.keys():
    for plot in plots_dir_dict[pdb].keys():
        fig = plots_dir_dict[pdb][plot]
        fig.show()

## Types of frustration modes the Frustratometer you can calculate in this notebook:

| Frustration Mode | Description | Mathematical Formula | Example Calculation |
|------------------|-------------|----------------------|---------------------|
| Configurational  | Compares the native energy of each contact in the protein to a set of decoy energies from random variants of the same contact. A contact is considered frustrated if its native energy is higher than the average of the decoys. | $F_c = \frac{E_n - \langle E_d \rangle}{\sigma_d}$ <br><br> $E_n$ = native energy of contact <br> $\langle E_d \rangle$ = mean energy of decoys <br> $\sigma_d$ = standard deviation of decoy energies | Native contact energy $E_n = -2.5$ <br> Mean decoy energy $\langle E_d \rangle = -5.2$ <br> Decoy std dev $\sigma_d = 1.8$ <br><br> $F_c = \frac{-2.5 - (-5.2)}{1.8} = 1.5$ <br><br> $F_c > 0$, so contact is frustrated |
| Mutational       | Compares the native energy of each contact to the average energy of all possible mutations of the amino acids forming that contact. A contact is considered frustrated if mutating it makes the energy more favorable on average. | $F_m = \frac{E_n - \langle E_m \rangle}{\sigma_m}$ <br><br> $E_n$ = native energy of contact <br> $\langle E_m \rangle$ = mean energy of all mutations <br> $\sigma_m$ = standard deviation of mutation energies | Native contact energy $E_n = -4.2$ <br> Mean mutation energy $\langle E_m \rangle = -6.8$ <br> Mutation std dev $\sigma_m = 2.1$ <br><br> $F_m = \frac{-4.2 - (-6.8)}{2.1} = 1.2$ <br><br> $F_m > 0$, so contact is frustrated |  
| Single Residue   | Calculates the total frustration of all contacts a single residue is involved in. Residues with many frustrated contacts are considered highly frustrated. | $F_r = \frac{1}{N} \sum_{i=1}^N F_{c,i}$ <br><br> $F_{c,i}$ = configurational frustration of $i$th contact <br> $N$ = total number of contacts residue is involved in | Residue involved in 3 contacts: <br> $F_{c,1} = 0.8$ <br> $F_{c,2} = 1.2$ <br> $F_{c,3} = -0.5$ <br><br> $F_r = \frac{1}{3}(0.8 + 1.2 + -0.5) = 0.5$ <br><br> $F_r > 0$, so residue is net frustrated |

In a nuthshell:
- Configurational frustration compares native contact energy to decoys
- Mutational frustration compares native contact energy to average mutation energy  
- Single residue frustration averages configurational frustration over all of a residue's contacts

The key equations are:

$F_c = \frac{E_n - \langle E_d \rangle}{\sigma_d}$ (configurational)

$F_m = \frac{E_n - \langle E_m \rangle}{\sigma_m}$ (mutational)  

$F_r = \frac{1}{N} \sum_{i=1}^N F_{c,i}$ (single residue)

Where $E_n$ is the native energy, $\langle E_d \rangle$ and $\langle E_m \rangle$ are mean decoy and mutation energies, and $\sigma_d$ and $\sigma_m$ are the standard deviations of the decoy and mutation energy distributions.

